# SI'26 – Week 1: Urdu OCR Dataset Collection
**qandeel asim**
## Dataset Preparation Overview

The purpose of this notebook is to build a small Urdu text recognition dataset using three different data sources:

1. **UTRSet-Real** – A publicly available research dataset containing real printed Urdu text, downloaded from Google Drive.
2. **Synthetic Images** – Artificially generated Urdu text images created by rendering custom Urdu sentences using the Noto Nastaliq Urdu font.
3. **Manual Screenshots** – Screenshots of Urdu text collected manually from sources such as Dawn Urdu, BBC Urdu, Jang, or Urdu Wikipedia.

All images are stored in the `data/raw/<category>/` directories, while the corresponding ground-truth text for each image is recorded in `data/labels.csv` using the columns `image` and `text`.



## 0. Setup — Imports & Install

In [1]:
!pip install Pillow arabic-reshaper python-bidi gdown -q
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 6.5 MB/s eta 0:00:00
Dependencies installed.


In [2]:
import csv
import os
import urllib.request
import zipfile
import subprocess

from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display

## 1. Folder Structure
Pehle apne dataset ke liye folders bana lete hain, har source/category ke liye ek folder.

In [3]:
FOLDERS = [
    'data/raw/newspaper',
    'data/raw/books',
    'data/raw/signboards',
    'data/raw/synthetic',
    'data/raw/other',
]

LABELS_CSV = 'data/labels.csv'

for folder in FOLDERS:
    os.makedirs(folder, exist_ok=True)
    print(f'Created: {folder}')

print('Folder structure ready!')

Created: data/raw/newspaper
Created: data/raw/books
Created: data/raw/signboards
Created: data/raw/synthetic
Created: data/raw/other
Folder structure ready!


## 2. Helper — Save Rows to `labels.csv`
This function safely updates the labels.csv file by merging newly generated entries with the existing records instead of overwriting them. It ensures that previously saved labels are preserved, while only new image–text pairs are appended. As a result, the notebook can be executed multiple times without creating duplicate entries or losing any existing annotations.



In [4]:
def append_to_labels_csv(new_rows, labels_csv=LABELS_CSV):
    """Append new_rows (list of dicts with 'image' and 'text') to labels_csv,
    keeping any rows already present."""
    existing_rows = []
    if os.path.exists(labels_csv):
        with open(labels_csv, 'r', encoding='utf-8') as f:
            existing_rows = list(csv.DictReader(f))

    all_rows = existing_rows + new_rows
    os.makedirs(os.path.dirname(labels_csv), exist_ok=True)
    with open(labels_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['image', 'text'])
        writer.writeheader()
        writer.writerows(all_rows)

    print(f'labels.csv now has {len(all_rows)} total entries '
          f'(+{len(new_rows)} new).')
    return all_rows

## 3. Source 1 — UTRSet-Real (Public Dataset)
UTRSet-Real Dataset

UTRSet-Real is a printed Urdu text dataset released alongside the UTRNet research paper (ICDAR 2023). In this notebook, the dataset is downloaded as a ZIP file from Google Drive, extracted, and the ground-truth annotation file (gt.txt) is located. A predefined number of samples (N_SAMPLES) are then copied to the data/raw/other/ directory, and their corresponding ground-truth text is recorded in data/labels.csv.

Reference:
Rahman, A., Ghosh, A., & Arora, C. (2023). UTRNet: High-Resolution Urdu Text Recognition in Printed Documents. Proceedings of ICDAR 2023, Springer Nature Switzerland.

License:
CC BY-NC-SA 4.0 – This dataset is intended for non-commercial and research purposes only.

In [5]:
# --- Config ---
GDRIVE_FILE_ID = "1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T"
DOWNLOAD_DIR = "utrset_real_download"
ZIP_PATH = os.path.join(DOWNLOAD_DIR, "utrset_real.zip")
EXTRACT_DIR = os.path.join(DOWNLOAD_DIR, "extracted")
OUT_DIR = "data/raw/other"
N_SAMPLES = 60  # kitni images is dataset se lene hain

In [6]:
def download_utrset():
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    if os.path.exists(ZIP_PATH):
        print(f"Already downloaded: {ZIP_PATH}")
        return

    print("Downloading UTRSet-Real from Google Drive (hundreds of MB, may take a few minutes)...")
    try:
        import gdown
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)
    except ImportError:
        subprocess.run(["gdown", "--id", GDRIVE_FILE_ID, "-O", ZIP_PATH], check=True)

    if not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) == 0:
        raise RuntimeError(
            "Download failed or produced an empty file. Google Drive sometimes blocks "
            "automated downloads of large files with a warning page instead of the real file. "
            "Agar yeh ho, toh link manually browser mein khol kar download karein: "
            f"https://drive.google.com/file/d/{GDRIVE_FILE_ID}/view"
        )


def extract_utrset():
    if os.path.exists(EXTRACT_DIR) and os.listdir(EXTRACT_DIR):
        print(f"Already extracted: {EXTRACT_DIR}")
        return
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extracted.")

In [7]:
def find_gt_file(root):
    """Locate the ground-truth label file inside the extracted dataset."""
    candidates = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower() in ("gt.txt", "ground_truth.txt", "labels.txt"):
                candidates.append(os.path.join(dirpath, fname))
    return candidates


def find_images(root):
    images = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                images.append(os.path.join(dirpath, fname))
    return images

In [8]:
def collect_utrset_samples():
    """Download + extract UTRSet-Real, then copy up to N_SAMPLES (image, text)
    pairs into OUT_DIR. Returns the list of new rows for labels.csv."""
    download_utrset()
    extract_utrset()

    gt_files = find_gt_file(EXTRACT_DIR)
    images = find_images(EXTRACT_DIR)
    print(f"Found {len(gt_files)} ground-truth file(s), {len(images)} image(s) in the extracted archive.")

    os.makedirs(OUT_DIR, exist_ok=True)
    new_rows = []

    if not gt_files:
        print("No ground-truth file found, skipping label extraction.")
        return new_rows

    gt_path = gt_files[0]
    print(f"Parsing ground truth from: {gt_path}")
    with open(gt_path, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    gt_dir = os.path.dirname(gt_path)
    for line in lines:
        if len(new_rows) >= N_SAMPLES:
            break
        if "\t" in line:
            img_rel_path, text = line.split("\t", 1)
        elif " " in line:
            img_rel_path, text = line.split(" ", 1)
        else:
            continue

        src_img_path = os.path.join(gt_dir, img_rel_path)
        if not os.path.exists(src_img_path):
            continue

        dst_name = f"utrset_{len(new_rows)+1:03d}{os.path.splitext(src_img_path)[1]}"
        dst_img_path = os.path.join(OUT_DIR, dst_name)
        with open(src_img_path, "rb") as fin, open(dst_img_path, "wb") as fout:
            fout.write(fin.read())

        new_rows.append({"image": dst_img_path, "text": text})

    print(f"Copied {len(new_rows)} UTRSet-Real samples into {OUT_DIR}.")
    return new_rows

In [9]:
utrset_rows = collect_utrset_samples()
append_to_labels_csv(utrset_rows)

Downloading...
From (original): https://drive.google.com/uc?id=1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T
From (redirected): https://drive.google.com/uc?id=1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T&confirm=t&uuid=1ffae24d-19db-4310-851f-874d980b07bd
To: /content/utrset_real_download/utrset_real.zip
100%|██████████| 212M/212M [00:02<00:00, 73.0MB/s]


Extracting...
Extracted.
Found 2 ground-truth file(s), 22322 image(s) in the extracted archive.
Parsing ground truth from: utrset_real_download/extracted/UTRSet-Real/train/gt.txt
Copied 60 UTRSet-Real samples into data/raw/other.
labels.csv now has 60 total entries (+60 new).


[{'image': 'data/raw/other/utrset_001.jpg',
  'text': 'اندراج و تحریر شرعاً صرف مستحب اور پسندیدہ ہے وہ واجب نہیں کہ کسی شرعی'},
 {'image': 'data/raw/other/utrset_002.jpg',
  'text': 'اور ہندوستان کی مسلم ریاستوں میں اس پر کم و بیش عمل درآمد بھی ہے، جہاں تک جبری'},
 {'image': 'data/raw/other/utrset_003.jpg',
  'text': 'بلکہ اس کے نزدیک وہ پسندیدہ اور مستحسن بھی ہے، عملی پہلوکے لحاظ سے اسلامی سلطنتوں'},
 {'image': 'data/raw/other/utrset_004.jpg',
  'text': 'جہاں تک پہلی صورت کاتعلق ہے، شریعت اسلامیہ کی رو سے اس پر اعتراض جائز نہیں،'},
 {'image': 'data/raw/other/utrset_005.jpg',
  'text': 'ہے، اس لیے اس کولازمی طورپر جاری کردیاجائے، اس کانام جبر ی اندراج ہے،'},
 {'image': 'data/raw/other/utrset_006.jpg',
  'text': 'مقدمات سے نجات مل سکتی ہے، نکاح کے ثبوت اورک دین مہر کے تعین میں سہولت ہوتی'},
 {'image': 'data/raw/other/utrset_007.jpg',
  'text': 'دوسرا پہلو یہ ہے کہ چونکہ اس اندراج میں بہت سی مصلحتیں ہیں، اس سے بڑی حدتک'},
 {'image': 'data/raw/other/utrset_008.jpg',
  'text': 'جس قدر لوگ

## 4. Source 2 — Synthetic Urdu Images

This section creates synthetic Urdu text images by rendering a collection of manually written Urdu sentences using the Noto Nastaliq Urdu font. The arabic_reshaper library is used to correctly reshape Urdu/Arabic characters into their connected forms, while the python-bidi library ensures that the text is displayed in the proper right-to-left (RTL) direction. The generated images are saved for use in the dataset, and their corresponding text annotations are added to data/labels.csv.



In [10]:
urdu_texts = [
    "پاکستان زندہ باد",
    "آج کا موسم خوشگوار ہے",
    "تعلیم ہر انسان کا حق ہے",
    "کراچی پاکستان کا سب سے بڑا شہر ہے",
    "محنت کامیابی کی کنجی ہے",
    "علم انسان کو روشنی دیتا ہے",
    "وقت کی قدر کرنی چاہیے",
    "صحت سب سے بڑی نعمت ہے",
    "دوستی ایک قیمتی رشتہ ہے",
    "محنت کبھی رائیگاں نہیں جاتی",
]

In [11]:
FONT_PATH = "NotoNastaliqUrdu-Regular.ttf"
if not os.path.exists(FONT_PATH):
    font_url = (
        "https://raw.githubusercontent.com/google/fonts/main/ofl/"
        "notonastaliqurdu/NotoNastaliqUrdu%5Bwght%5D.ttf"
    )
    try:
        urllib.request.urlretrieve(font_url, FONT_PATH)
        print(f"Downloaded font to {FONT_PATH}")
    except Exception as e:
        print(f"Could not auto-download font ({e}).")
        print("Manually download 'Noto Nastaliq Urdu' from fonts.google.com,")
        print(f"upload it to Colab's file panel, named {FONT_PATH}")

font = ImageFont.truetype(FONT_PATH, 36)

Downloaded font to NotoNastaliqUrdu-Regular.ttf


In [12]:
def generate_synthetic_images():
    rows = []
    for i, text in enumerate(urdu_texts):
        reshaped = arabic_reshaper.reshape(text)
        bidi_text = get_display(reshaped)

        dummy_img = Image.new('RGB', (10, 10))
        dummy_draw = ImageDraw.Draw(dummy_img)
        bbox = dummy_draw.textbbox((0, 0), bidi_text, font=font)
        text_w = bbox[2] - bbox[0]
        text_h = bbox[3] - bbox[1]

        img = Image.new('RGB', (text_w + 40, text_h + 40), color='white')
        draw = ImageDraw.Draw(img)
        draw.text((20 - bbox[0], 20 - bbox[1]), bidi_text, fill='black', font=font)

        save_path = f'data/raw/synthetic/urdu_{i+1}.png'
        img.save(save_path)
        rows.append({'image': save_path, 'text': text})

    print(f'Done! {len(rows)} synthetic images in data/raw/synthetic/')
    return rows

In [13]:
synthetic_rows = generate_synthetic_images()
append_to_labels_csv(synthetic_rows)

Done! 10 synthetic images in data/raw/synthetic/
labels.csv now has 70 total entries (+10 new).


[{'image': 'data/raw/other/utrset_001.jpg',
  'text': 'اندراج و تحریر شرعاً صرف مستحب اور پسندیدہ ہے وہ واجب نہیں کہ کسی شرعی'},
 {'image': 'data/raw/other/utrset_002.jpg',
  'text': 'اور ہندوستان کی مسلم ریاستوں میں اس پر کم و بیش عمل درآمد بھی ہے، جہاں تک جبری'},
 {'image': 'data/raw/other/utrset_003.jpg',
  'text': 'بلکہ اس کے نزدیک وہ پسندیدہ اور مستحسن بھی ہے، عملی پہلوکے لحاظ سے اسلامی سلطنتوں'},
 {'image': 'data/raw/other/utrset_004.jpg',
  'text': 'جہاں تک پہلی صورت کاتعلق ہے، شریعت اسلامیہ کی رو سے اس پر اعتراض جائز نہیں،'},
 {'image': 'data/raw/other/utrset_005.jpg',
  'text': 'ہے، اس لیے اس کولازمی طورپر جاری کردیاجائے، اس کانام جبر ی اندراج ہے،'},
 {'image': 'data/raw/other/utrset_006.jpg',
  'text': 'مقدمات سے نجات مل سکتی ہے، نکاح کے ثبوت اورک دین مہر کے تعین میں سہولت ہوتی'},
 {'image': 'data/raw/other/utrset_007.jpg',
  'text': 'دوسرا پہلو یہ ہے کہ چونکہ اس اندراج میں بہت سی مصلحتیں ہیں، اس سے بڑی حدتک'},
 {'image': 'data/raw/other/utrset_008.jpg',
  'text': 'جس قدر لوگ

## 5. Source 3 — Manual Screenshots (Step 6, 30+ images)

Source 3 — Manual Screenshots (Step 6, 30+ Images)

This data source is collected manually rather than generated automatically.

Capture screenshots of Urdu text from reliable sources such as Dawn Urdu (urdu.dawn.com), BBC Urdu (bbc.com/urdu), Jang (jang.com.pk), or Urdu Wikipedia. Crop each screenshot tightly so that it contains only the text region.
In Google Colab, open the Files panel from the left sidebar, navigate to the data/raw/newspaper/ directory (or another appropriate category such as books/, signboards/, or other/), and upload the screenshots by dragging and dropping them into the folder.
In the provided notebook cell, enter the correct ground-truth text corresponding to each uploaded image, then run the cell to append the image–text pairs to data/labels.csv.

A minimum of 30 manually collected images is required from this source to ensure sufficient dataset diversity.

In [14]:
# Example — har row mein image ka path aur uska Urdu text daalein:
manual_rows = [
    # {'image': 'data/raw/newspaper/screenshot_1.png', 'text': 'یہاں متن لکھیں'},
    # {'image': 'data/raw/newspaper/screenshot_2.png', 'text': 'یہاں متن لکھیں'},
]

if manual_rows:
    append_to_labels_csv(manual_rows)
else:
    print('manual_rows abhi khali hai — screenshots add karne ke baad upar wali list bharein.')

manual_rows abhi khali hai — screenshots add karne ke baad upar wali list bharein.


## 6. Final Check
Final `labels.csv` ka overview — kitne rows total hain aur top kuch rows kaisi dikhti hain.

In [15]:
with open(LABELS_CSV, 'r', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

print(f'Total labeled entries: {len(rows)}')
for r in rows[:5]:
    print(r)

Total labeled entries: 70
{'image': 'data/raw/other/utrset_001.jpg', 'text': 'اندراج و تحریر شرعاً صرف مستحب اور پسندیدہ ہے وہ واجب نہیں کہ کسی شرعی'}
{'image': 'data/raw/other/utrset_002.jpg', 'text': 'اور ہندوستان کی مسلم ریاستوں میں اس پر کم و بیش عمل درآمد بھی ہے، جہاں تک جبری'}
{'image': 'data/raw/other/utrset_003.jpg', 'text': 'بلکہ اس کے نزدیک وہ پسندیدہ اور مستحسن بھی ہے، عملی پہلوکے لحاظ سے اسلامی سلطنتوں'}
{'image': 'data/raw/other/utrset_004.jpg', 'text': 'جہاں تک پہلی صورت کاتعلق ہے، شریعت اسلامیہ کی رو سے اس پر اعتراض جائز نہیں،'}
{'image': 'data/raw/other/utrset_005.jpg', 'text': 'ہے، اس لیے اس کولازمی طورپر جاری کردیاجائے، اس کانام جبر ی اندراج ہے،'}
